In [115]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.datasets import make_classification
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

from scipy.stats import shapiro
from scipy.spatial.distance import euclidean
from scipy.stats import shapiro


data = pd.read_csv(
    "M:/accenture/ds-ml-training/ds-ai-assignments-maksym-bondar/weeks/week1/data/day.csv"
)

data["dteday"] = pd.to_datetime(data["dteday"])

data.head()


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [116]:
data["dow"] = data["dteday"].apply(lambda x: x.date().weekday())
data["is_weekend"] = data["dteday"].apply(
    lambda x: 1 if x.date().weekday() in (5, 6) else 0
)

data[["dteday", "weekday", "dow", "is_weekend"]].head()


,dteday,weekday,dow,is_weekend
0,2011-01-01,6,5,1
1,2011-01-02,0,6,1
2,2011-01-03,1,0,0
3,2011-01-04,2,1,0
4,2011-01-05,3,2,0


In [118]:
def make_harmonic_features(value, period=24):
    value *= 2 * np.pi / period
    return np.cos(value), np.sin(value)

print(euclidean(make_harmonic_features(23), make_harmonic_features(1)))
print(euclidean(make_harmonic_features(9), make_harmonic_features(11)))
print(euclidean(make_harmonic_features(9), make_harmonic_features(21)))

data["mnth_cos"], data["mnth_sin"] = make_harmonic_features(data["mnth"], period=12)
data["weekday_cos"], data["weekday_sin"] = make_harmonic_features(data["dow"], period=7)

data[["mnth", "mnth_cos", "mnth_sin", "dow", "weekday_cos", "weekday_sin"]].head()


0.5176380902050424
0.5176380902050414
2.0


,mnth,mnth_cos,mnth_sin,dow,weekday_cos,weekday_sin
0,1,0.962654,0.270734,5,-0.222521,-0.974928
1,1,0.962654,0.270734,6,0.623490,-0.781831
2,1,0.962654,0.270734,0,1.000000,0.000000
3,1,0.962654,0.270734,1,0.623490,0.781831
4,1,0.962654,0.270734,2,-0.222521,0.974928


In [119]:
cnt = data["cnt"].astype(float)
cnt_reshaped = cnt.values.reshape(-1, 1)

print("Shapiro original cnt:", shapiro(cnt_reshaped[:500]))
cnt_log = np.log(cnt + 1)
print("Shapiro log(cnt+1):", shapiro(cnt_log.values[:500].reshape(-1, 1)))

cnt_z = StandardScaler().fit_transform(cnt_reshaped).flatten()

cnt_mm = MinMaxScaler().fit_transform(cnt_reshaped).flatten()

print("StandardScaler example:", cnt_z[:5])
print("MinMaxScaler example:", cnt_mm[:5])
print("Log transform example:", cnt_log[:5].values)


Shapiro original cnt: ShapiroResult(statistic=np.float64(0.9841503102910155), pvalue=np.float64(2.8701103934155135e-05))
Shapiro log(cnt+1): ShapiroResult(statistic=np.float64(0.911219906607336), pvalue=np.float64(1.6059272373298034e-16))
StandardScaler example: [-1.81795256 -1.91299949 -1.62992496 -1.51989782 -1.50026856]
MinMaxScaler example: [0.11079153 0.08962264 0.15266912 0.17717441 0.18154625]
Log transform example: [6.89365635 6.68710861 7.20785987 7.35436233 7.37838371]


In [120]:
x_data_generated, y_data_generated = make_classification()

print("Shapes with VarianceThreshold:")
print("thr=0.7:", VarianceThreshold(0.7).fit_transform(x_data_generated).shape)
print("thr=0.8:", VarianceThreshold(0.8).fit_transform(x_data_generated).shape)
print("thr=0.9:", VarianceThreshold(0.9).fit_transform(x_data_generated).shape)

x_data_kbest = SelectKBest(f_classif, k=5).fit_transform(
    x_data_generated, y_data_generated
)
x_data_varth = VarianceThreshold(0.9).fit_transform(x_data_generated)

logit = LogisticRegression(solver="lbfgs", random_state=17, max_iter=1000)

print("\nneg_log_loss scores:")
print("LR all features:",
      cross_val_score(
          logit, x_data_generated, y_data_generated,
          scoring="neg_log_loss", cv=5
      ).mean())

print("LR SelectKBest:",
      cross_val_score(
          logit, x_data_kbest, y_data_generated,
          scoring="neg_log_loss", cv=5
      ).mean())

print("LR VarThreshold:",
      cross_val_score(
          logit, x_data_varth, y_data_generated,
          scoring="neg_log_loss", cv=5
      ).mean())


Shapes with VarianceThreshold:
thr=0.7: (100, 20)
thr=0.8: (100, 20)
thr=0.9: (100, 13)

neg_log_loss scores:
LR all features: -0.27408172783424517
LR SelectKBest: -0.2311629097234456
LR VarThreshold: -0.25016120185469937


In [121]:
rf = RandomForestClassifier(n_estimators=10, random_state=17)
pipe = make_pipeline(SelectFromModel(estimator=rf), logit)

print("LR all:",
      cross_val_score(
          logit, x_data_generated, y_data_generated,
          scoring="neg_log_loss", cv=5
      ).mean())

print("RF all:",
      cross_val_score(
          rf, x_data_generated, y_data_generated,
          scoring="neg_log_loss", cv=5
      ).mean())

print("SelectFromModel(RF) - LR:",
      cross_val_score(
          pipe, x_data_generated, y_data_generated,
          scoring="neg_log_loss", cv=5
      ).mean())

pipe1 = make_pipeline(StandardScaler(), SelectFromModel(estimator=rf), logit)
pipe2 = make_pipeline(StandardScaler(), logit)

print("\nLR + selection:",
      cross_val_score(pipe1, x_data_generated, y_data_generated,
                      scoring="neg_log_loss", cv=5).mean())
print("LR:",
      cross_val_score(pipe2, x_data_generated, y_data_generated,
                      scoring="neg_log_loss", cv=5).mean())
print("RF:",
      cross_val_score(rf, x_data_generated, y_data_generated,
                      scoring="neg_log_loss", cv=5).mean())


LR all: -0.27408172783424517
RF all: -0.9789293009642727
SelectFromModel(RF) - LR: -0.24037490704261016

LR + selection: -0.23995928447003473
LR: -0.26815213968858925
RF: -0.9789293009642727
